to build a full prototype for the technical question/answerer.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. 

see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.


steps:
1. create separate openai and claude function to call
2. Add the tools and handler
3. Call the function to switch between gpt and Claude
4. Add Gradio UI
5. Add Streaming


In [ ]:
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from IPython.display import Markdown, display, update_display
from openai import OpenAI 
import gradio as gr
from anthropic import Anthropic

In [ ]:
# constants

MODEL_GPT = 'gpt-4.1-nano'
MODEL_CLAUDE = 'claude-sonnet-4-5-20250929'

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")


In [ ]:
openai = OpenAI()
anthropic = Anthropic()


Define the TOOL (Pure Python)

This is the heart of your assistant.

Question to answer:

What is the minimum useful capability your assistant must have?

For a technical tutor, the most basic tool is:

Explain a technical concept at a given level

In [ ]:
def explain_tech_concept(concept: str, level: str = "beginner") -> str:
    """
    Explain a technical concept in simple terms.
    
    Parameters:
    - topic: the technical topic to explain
    - level: beginner | intermediate | advanced
    
    Returns:
    - A clear explanation as text
    """
    
    if level == "beginner":
        return f"{concept} is a concept that helps you understand the basics in a simple way."
    
    elif level == "intermediate":
        return f"{concept} involves some deeper ideas and is commonly used in real projects."
    
    elif level == "advanced":
        return f"{concept} includes complex details and is optimized for performance and scale."
    
    else:
        return "Invalid level provided."


In [ ]:
print(explain_tech_concept("Python decorators", "beginner"))
print(explain_tech_concept("Python decorators", "advanced"))


TOOL SCHEMAS ( HOW LLM UNDERTANDS)

In [ ]:
gpt_tools = [
    {
        "type": "function",
        "function": {
            "name": "explain_tech_concept",
            "description": "Explain a technical concept",
            "parameters": {
                "type": "object",
                "properties": {
                    "concept": {"type": "string"},
                    "level": {
                        "type": "string",
                        "enum": ["beginner", "intermediate", "advanced"]
                    }
                },
                "required": ["topic"]
            }
        }
    }
]


In [ ]:
claude_tools = [
    {
        "name": "explain_tech_concept",
        "description": "Explain a technical concept at a given level",
        "input_schema": {
            "type": "object",
            "properties": {
                "concept": {
                    "type": "string",
                    "description": "The technical concept to explain"
                },
                "level": {
                    "type": "string",
                    "description": "beginner | intermediate | advanced"
                }
            },
            "required": ["concept", "level"]
        }
    }
]


Tool handler (Bridge between LLM & Python)

LLMs cannot execute code.
They only request execution.

In [ ]:
def handle_tool_call(tool_name, arguments):
    if tool_name == "explain_tech_concept":
        return explain_tech_concept(**arguments)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


GPT wrapper function

Handles:

Messages

Tool calls

Tool results

Final answer

In [ ]:
def ask_gpt(messages, system_prompt, tools=None):
    response = openai.chat.completions.create(
        model= MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            *messages
        ],
        tools=tools
    )

    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        tool_result = handle_tool_call(
        tool_call.function.name,
        json.loads(tool_call.function.arguments)
    )

    # append as assistant, not tool
    messages.append({"role": "assistant", "content": tool_result})

    final_response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            *messages
        ]
    )
    return final_response.choices[0].message.content


In [ ]:
messages = [
    {"role": "user", "content": "Explain Python decorators using the explain_tech_concept tool at beginner level."}
]

system_prompt = "You are a technical tutor. Use the tool `explain_tech_concept` to answer technical questions."

print(ask_gpt(messages, system_prompt, tools=gpt_tools))



Claude wrapper function

In [ ]:
def ask_claude(messages, system_prompt, tools=None):
    kwargs = {
        "model": MODEL_CLAUDE,
        "max_tokens": 1024,
        "system": system_prompt,
        "messages": messages,
    }

    if tools:
        kwargs["tools"] = tools

    response = anthropic.messages.create(**kwargs)

    final_text = ""

    for block in response.content:
        # 🔧 TOOL CALL
        if block.type == "tool_use":
            tool_result = handle_tool_call(block.name, block.input)

            messages.append({
                "role": "assistant",
                "content": response.content
            })

            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": tool_result
                    }
                ]
            })

            followup = anthropic.messages.create(
                model=MODEL_CLAUDE,
                max_tokens=1024,
                system=system_prompt,
                messages=messages
            )

            return followup.content[0].text

        # 📝 NORMAL TEXT
        if block.type == "text":
            final_text += block.text

    return final_text


In [ ]:
messages = [
    {"role": "user", "content": "Explain decorators in Python for a beginner"}
]

system_prompt = "You are a helpful technical tutor."

response = ask_claude(
    messages=messages,
    system_prompt=system_prompt,
    tools=claude_tools
)

print(response)

LLM router (Model switching)

In [ ]:
def run_llm(model_name, messages, system_prompt, tools=None):
    if model_name == "gpt":
        return ask_gpt(messages, system_prompt, tools)

    elif model_name == "claude":
        return ask_claude(messages, system_prompt, tools)

    else:
        raise ValueError(f"Unsupported model: {model_name}")




In [ ]:
messages = [
    {
        "role": "user",
        "content": "Explain decorators in Python for a beginner"
    }
]

system_prompt = "You are a technical tutor."

print(run_llm("gpt", messages.copy(), system_prompt, tools = gpt_tools))
print(run_llm("claude", messages.copy(), system_prompt))


Creating GRADIO UI

In [ ]:
def chat_fn(user_input, history, model_choice):
    messages = history + [{"role": "user", "content": user_input}]
    system_prompt = "You are a helpful technical tutor."

    reply = run_llm(model_choice, messages, system_prompt)

    history.append({"role": "user", "content": user_input})
    history.append({"role": "assistant", "content": reply})

    return history, history

with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Technical Q&A Assistant")

    model_choice = gr.Radio(
        ["gpt", "claude"],
        value="gpt",
        label="Choose Model"
    )

    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox()
    state = gr.State([])

    msg.submit(chat_fn, [msg, state, model_choice], [chatbot, state])

demo.launch()


# Using Streaming and AUDIO

In [ ]:
import os
import uuid

def talker(message):
    os.makedirs("audio", exist_ok=True)

    audio_path = f"audio/{uuid.uuid4()}.mp3"

    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message
    )

    # Save audio to file
    with open(audio_path, "wb") as f:
        f.write(response.content)

    return audio_path   # ✅ Gradio wants a file path